# Dual-window synthesis tradeoffs

This notebook is the slower companion to `notes/dual-windows-do-not-make-conditioning-free.md`.

The bounded question is simple: **once normalized same-window overlap-add is exact but poorly conditioned, does a flatter-looking dual window actually buy a calmer synthesis path?**


## 1. The reconstruction constraint

For one analysis window $g[n]$, one synthesis window $h[n]$, and hop $H$, the periodic overlap-add reconstruction constraint is

$$\sum_m g[n-mH] h[n-mH] = 1.$$

In this repo's painless periodic setting, the so-called normalized same-window path already gives the **canonical dual**. So the real comparison is not same-window versus canonical. It is canonical versus a dual that is forced to look as constant as possible while still satisfying the same exact reconstruction rule.


In [ ]:
from windowlab.reconstruct import compare_dual_windows
from windowlab.windows import build_window

cases = [
    ('hann', 64),
    ('hann', 32),
    ('blackman-harris', 64),
    ('blackman-harris', 32),
    ('flattop', 64),
    ('flattop', 32),
]

rows = []
for name, hop in cases:
    comparison = compare_dual_windows(build_window(name, 128), hop)
    rows.append({
        'window': name,
        'hop': hop,
        'canonical_flatness': round(comparison.canonical.relative_constant_rmse, 3),
        'constant_flatness': round(comparison.closest_constant.relative_constant_rmse, 3),
        'canonical_rms_noise': round(comparison.canonical.rms_noise_gain, 3),
        'constant_rms_noise': round(comparison.closest_constant.rms_noise_gain, 3),
        'canonical_worst_noise': round(comparison.canonical.worst_noise_gain, 3),
        'constant_worst_noise': round(comparison.closest_constant.worst_noise_gain, 3),
    })

rows


## 2. The sharper read

The constant-looking dual does exactly what its name promises: it becomes much flatter than the canonical dual.

But the canonical dual is still the calmer minimum-energy answer in these cases. That is why its coefficient-noise gain stays lower across the whole packet.

This is the useful fork in the road:

- if you want a synthesis window that *looks* more COLA-like, the constant-looking dual can do that
- if you want the bounded low-noise answer, the canonical dual already has the better claim
- if the case is still ugly, shrinking hop is often the honest fix


In [ ]:
worst_tradeoff = max(
    rows,
    key=lambda row: row['constant_rms_noise'] / row['canonical_rms_noise'],
)
worst_tradeoff


For this pass, that worst tradeoff lands on flat-top at quarter-hop. The flatter dual is visibly tidier, but it also raises the RMS coefficient-noise gain sharply.

## 3. Problems worth trying next

1. Compare the canonical dual against one second desired target only if it changes the story instead of just proving the same point twice.
2. Keep the same dual-window comparison and try one modified-coefficient toy example only if it sharpens the framing story instead of widening the repo into generic STFT folklore.
3. Repeat the same pass at one second frame length only if the pressure cases move enough to matter.

## References

- SciPy `ShortTimeFFT` and dual-window documentation
- SciPy `closest_STFT_dual_window`
- Julius O. Smith, *Spectral Audio Signal Processing*
